# 03 - Smoke Test Khung Huấn Luyện & Kiểm Chứng Cơ Chế Resume

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (COVID-19 Radiography Database).  
**Tuần 3 (Sprint 2) - Track B:** Dựng khung huấn luyện, kiểm thử đơn vị, chạy smoke test trên GPU T4, kiểm chứng cơ chế ngắt phiên và tự động phục hồi (Resume) từ `last.pt`.  
**Người phụ trách chính:** TV2 (train.py), TV4 (augmentation), TV6 (metrics), TV1 (kèm cặp).  

## Mục tiêu kiểm chứng:
1. Xác nhận môi trường GPU CUDA (Tesla T4) và Mixed Precision (AMP).
2. Chạy toàn bộ các bài kiểm thử tự động (`pytest`):
   - `tests/test_split.py` (chống rò rỉ nhóm, tỷ lệ 70/15/15, cân bằng lớp, hash SHA256).
   - `tests/test_metrics.py` (Macro-F1, Balanced Accuracy, ma trận nhầm lẫn).
   - `tests/test_resume.py` (khôi phục trạng thái mô hình và optimizer, khóa tập test).
3. Chạy 1 epoch huấn luyện ban đầu trên GPU, lưu `last.pt` và `best.pt`.
4. Giả lập ngắt phiên và khôi phục (Resume) thành công ở Epoch 2.
5. Chứng minh cờ `allow_test=False` bảo vệ tuyệt đối tập Test không bị truy cập trái phép.


In [ ]:
# 1. Khởi tạo cấu trúc module src/ và tests/ trong môi trường thực thi Kaggle
import os
import sys
from pathlib import Path

os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs("outputs/checkpoints", exist_ok=True)

with open("src/__init__.py", "w", encoding="utf-8") as f: f.write("")
with open("tests/__init__.py", "w", encoding="utf-8") as f: f.write("")

with open("src/dataset.py", "w", encoding="utf-8") as f: f.write('"""Module dataset.py\n\nTrách nhiệm:\n- Quản lý nạp dữ liệu từ file split_v1.csv (phụ trách: TV1).\n- Định nghĩa lớp PyTorch CXRDataset cho bài toán phân loại 4 nhóm ảnh X-quang ngực.\n- Thiết kế Data Augmentation cho tập Train và Transform chuẩn hóa cho Validation (phụ trách: TV4).\n- Cài đặt cơ chế bảo vệ nghiêm ngặt: Khóa tập Test (allow_test=False).\n- Tính toán Class Weights hỗ trợ xử lý mất cân bằng lớp.\n"""\n\nimport os\nfrom pathlib import Path\nfrom typing import Callable, Dict, List, Optional, Tuple, Union\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\n\n# 4 lớp bài toán phân loại X-quang ngực\nCLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]\nLABEL_TO_IDX: Dict[str, int] = {name: idx for idx, name in enumerate(CLASS_NAMES)}\nIDX_TO_LABEL: Dict[int, str] = {idx: name for idx, name in enumerate(CLASS_NAMES)}\n\n# Chuẩn hóa ImageNet\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\nIMAGENET_STD = [0.229, 0.224, 0.225]\n\n\ndef get_transforms(\n    img_size: int = 224,\n    is_train: bool = True,\n) -> transforms.Compose:\n    """Tạo chuỗi biến đổi tiền xử lý và tăng cường dữ liệu ảnh X-quang.\n\n    Quy tắc y khoa nghiêm ngặt (TV4 phụ trách):\n    - TUYỆT ĐỐI KHÔNG sử dụng RandomHorizontalFlip (lật ngang).\n      Lý do: Tim người nằm ở lồng ngực bên trái; lật ngang làm đảo ngược vị trí tim và\n      bóng các cơ quan nội tạng (Situs Inversus nhân tạo), gây sai lệch đặc trưng y học.\n    - Train augmentation an toàn: Xoay nhẹ (+/- 7 độ), co giãn nhẹ (scale 0.95-1.05),\n      chỉnh sáng và tương phản nhẹ (+/- 10%).\n    - Validation: Chỉ resize và normalize.\n\n    Args:\n        img_size (int): Kích thước cạnh ảnh đầu ra (mặc định 224).\n        is_train (bool): True nếu áp dụng augmentation cho tập train, False cho val/test.\n\n    Returns:\n        transforms.Compose: Pipeline tiền xử lý của torchvision.\n    """\n    if is_train:\n        return transforms.Compose([\n            transforms.Resize((img_size, img_size)),\n            transforms.RandomRotation(degrees=(-7, 7)),\n            transforms.RandomAffine(\n                degrees=0,\n                translate=(0.02, 0.02),\n                scale=(0.95, 1.05),\n            ),\n            transforms.ColorJitter(brightness=0.1, contrast=0.1),\n            transforms.ToTensor(),\n            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),\n        ])\n    else:\n        return transforms.Compose([\n            transforms.Resize((img_size, img_size)),\n            transforms.ToTensor(),\n            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),\n        ])\n\n\nclass CXRDataset(Dataset):\n    """Lớp Dataset nạp ảnh X-quang ngực từ split_v1.csv.\n\n    Hỗ trợ cơ chế khóa tập test: mặc định cấm truy cập split=\'test\' (allow_test=False).\n    """\n\n    def __init__(\n        self,\n        split_df_or_path: Union[pd.DataFrame, str, Path],\n        split: str = "train",\n        data_root: Optional[Union[str, Path]] = None,\n        transform: Optional[Callable] = None,\n        allow_test: bool = False,\n    ) -> None:\n        """Khởi tạo CXRDataset.\n\n        Args:\n            split_df_or_path: Đường dẫn tới file split_v1.csv hoặc pandas DataFrame.\n            split (str): Phân vùng cần nạp (\'train\', \'val\', \'test\').\n            data_root: Thư mục gốc chứa ảnh (ví dụ /kaggle/input/.../COVID-19_Radiography_Dataset).\n            transform: Chuỗi biến đổi ảnh. Nếu None sẽ tự tạo theo get_transforms.\n            allow_test (bool): Cờ cho phép đọc tập test. Mặc định là False.\n        """\n        split_lower = split.lower().strip()\n        if split_lower not in {"train", "val", "test"}:\n            raise ValueError(f"Split không hợp lệ: \'{split}\'. Chỉ chấp nhận \'train\', \'val\', hoặc \'test\'.")\n\n        # Cơ chế khóa nghiêm ngặt tập Test\n        if split_lower == "test" and not allow_test:\n            raise PermissionError(\n                "TẬP TEST ĐÃ BỊ KHÓA NGHIÊM NGẶT!\\n"\n                "Theo quy tắc đạo đức nghiên cứu và chống rò rỉ dữ liệu của đồ án:\\n"\n                "- Không được phép truy cập, đánh giá hoặc tinh chỉnh siêu tham số trên tập test ở Tuần 3-9.\\n"\n                "- Mọi quyết định lựa chọn mô hình chỉ được thực hiện trên tập Validation.\\n"\n                "- Tập Test chỉ được mở DUY NHẤT một lần tại Tuần 10 trên mô hình tối ưu đã chốt."\n            )\n\n        if isinstance(split_df_or_path, (str, Path)):\n            df_full = pd.read_csv(split_df_or_path)\n        else:\n            df_full = split_df_or_path.copy()\n\n        self.df = df_full[df_full["split"] == split_lower].reset_index(drop=True)\n        self.split = split_lower\n        self.data_root = Path(data_root) if data_root is not None else None\n\n        if transform is not None:\n            self.transform = transform\n        else:\n            self.transform = get_transforms(img_size=224, is_train=(split_lower == "train"))\n\n    def __len__(self) -> int:\n        return len(self.df)\n\n    def _resolve_image_path(self, row: pd.Series) -> Path:\n        """Định vị đường dẫn ảnh tuyệt đối một cách linh hoạt."""\n        # 1. Thử qua data_root ghép với rel_image_path\n        if self.data_root is not None:\n            candidate = self.data_root / row["rel_image_path"]\n            if candidate.exists():\n                return candidate\n\n        # 2. Thử rel_image_path trực tiếp từ thư mục hiện tại\n        candidate = Path(row["rel_image_path"])\n        if candidate.exists():\n            return candidate\n\n        # 3. Thử image_path lưu trong CSV nếu có\n        if "image_path" in row and pd.notna(row["image_path"]):\n            candidate = Path(row["image_path"])\n            if candidate.exists():\n                return candidate\n\n        raise FileNotFoundError(\n            f"Không thể tìm thấy file ảnh: {row[\'image_id\']} (rel: {row[\'rel_image_path\']})"\n        )\n\n    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:\n        row = self.df.iloc[idx]\n        img_path = self._resolve_image_path(row)\n\n        with Image.open(img_path) as img:\n            # Luôn chuyển đổi sang RGB 3 kênh để tương thích các kiến trúc CNN\n            img = img.convert("RGB")\n            tensor_img = self.transform(img)\n\n        label_name = row["label"]\n        label_idx = LABEL_TO_IDX[label_name]\n\n        return tensor_img, label_idx\n\n\ndef compute_class_weights(\n    split_df_or_path: Union[pd.DataFrame, str, Path],\n    split: str = "train",\n    num_classes: int = 4,\n) -> torch.FloatTensor:\n    """Tính toán trọng số nghịch đảo tần suất lớp (Balanced Class Weights) cho Loss Function.\n\n    Công thức: w_c = N / (C * N_c)\n    Giúp giảm thiểu tác động tiêu cực của tình trạng mất cân bằng lớp (Normal chiếm ~48%).\n\n    Args:\n        split_df_or_path: Đường dẫn tới CSV hoặc DataFrame.\n        split (str): Mặc định tính trên tập \'train\'.\n        num_classes (int): Số lớp (4).\n\n    Returns:\n        torch.FloatTensor: Mảng trọng số 1D tensor [w_0, w_1, w_2, w_3].\n    """\n    if isinstance(split_df_or_path, (str, Path)):\n        df = pd.read_csv(split_df_or_path)\n    else:\n        df = split_df_or_path\n\n    df_subset = df[df["split"] == split.lower()]\n    total_samples = len(df_subset)\n\n    class_counts = df_subset["label"].value_counts().to_dict()\n    weights = np.zeros(num_classes, dtype=np.float32)\n\n    for cls_name, cls_idx in LABEL_TO_IDX.items():\n        count = class_counts.get(cls_name, 1)\n        weights[cls_idx] = total_samples / (num_classes * count)\n\n    return torch.tensor(weights, dtype=torch.float32)\n\n\ndef build_dataloaders(\n    split_csv_path: Union[str, Path],\n    data_root: Optional[Union[str, Path]] = None,\n    batch_size: int = 32,\n    num_workers: int = 2,\n    img_size: int = 224,\n    allow_test: bool = False,\n) -> Dict[str, DataLoader]:\n    """Khởi tạo DataLoaders cho các phân vùng huấn luyện và kiểm định.\n\n    Args:\n        split_csv_path: Đường dẫn tới file split_v1.csv.\n        data_root: Thư mục gốc chứa ảnh (nếu chạy trên Kaggle/Colab).\n        batch_size: Kích thước mini-batch.\n        num_workers: Số luồng nạp dữ liệu.\n        img_size: Kích thước ảnh đầu vào mô hình.\n        allow_test: Cờ cho phép nạp tập test (mặc định False).\n\n    Returns:\n        Dict[str, DataLoader]: Dict chứa \'train\', \'val\' (và \'test\' nếu allow_test=True).\n    """\n    train_ds = CXRDataset(\n        split_df_or_path=split_csv_path,\n        split="train",\n        data_root=data_root,\n        transform=get_transforms(img_size=img_size, is_train=True),\n        allow_test=False,\n    )\n\n    val_ds = CXRDataset(\n        split_df_or_path=split_csv_path,\n        split="val",\n        data_root=data_root,\n        transform=get_transforms(img_size=img_size, is_train=False),\n        allow_test=False,\n    )\n\n    dataloaders = {\n        "train": DataLoader(\n            train_ds,\n            batch_size=batch_size,\n            shuffle=True,\n            num_workers=num_workers,\n            pin_memory=torch.cuda.is_available(),\n        ),\n        "val": DataLoader(\n            val_ds,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=num_workers,\n            pin_memory=torch.cuda.is_available(),\n        ),\n    }\n\n    if allow_test:\n        test_ds = CXRDataset(\n            split_df_or_path=split_csv_path,\n            split="test",\n            data_root=data_root,\n            transform=get_transforms(img_size=img_size, is_train=False),\n            allow_test=True,\n        )\n        dataloaders["test"] = DataLoader(\n            test_ds,\n            batch_size=batch_size,\n            shuffle=False,\n            num_workers=num_workers,\n            pin_memory=torch.cuda.is_available(),\n        )\n\n    return dataloaders\n')
with open("src/models.py", "w", encoding="utf-8") as f: f.write('"""Module models.py\n\nTrách nhiệm:\n- Định nghĩa và khởi tạo kiến trúc mạng học sâu cho bài toán phân loại X-quang 4 lớp.\n- Hỗ trợ mô hình ResNet-18 (tối thiểu cho Smoke Test Tuần 3), mở rộng DenseNet-121 và EfficientNet-B0.\n- Cho phép khởi tạo không cần trọng số pretrained (pretrained=False) để chạy offline trên Kaggle khi tắt internet.\n- Thay thế classifier head tương ứng với 4 lớp phân loại.\n- Cung cấp tiện ích đóng băng (freeze) và mở khóa (unfreeze) backbone phục vụ transfer learning.\n\nGhi chú phân công:\n- Theo docs/OWNERS.md, src/models.py hiện CHƯA có chủ sở hữu chính thức (được báo cáo lên Leader).\n"""\n\nfrom typing import Optional\nimport torch\nimport torch.nn as nn\nfrom torchvision import models\n\n\ndef get_model(\n    model_name: str = "resnet18",\n    num_classes: int = 4,\n    pretrained: bool = False,\n) -> nn.Module:\n    """Khởi tạo mô hình CNN và thay thế classifier head cho bài toán 4 lớp.\n\n    Args:\n        model_name (str): Tên kiến trúc (\'resnet18\', \'densenet121\', \'efficientnet_b0\').\n        num_classes (int): Số lượng lớp phân loại (mặc định 4).\n        pretrained (bool): True để tải trọng số ImageNet (yêu cầu internet hoặc file weights có sẵn).\n                           False để khởi tạo trọng số ngẫu nhiên (dùng cho Smoke Test với internet=False).\n\n    Returns:\n        nn.Module: Mô hình PyTorch đã điều chỉnh classifier head.\n    """\n    name_clean = model_name.lower().replace("-", "").replace("_", "")\n\n    if "resnet18" in name_clean:\n        weights = models.ResNet18_Weights.DEFAULT if pretrained else None\n        model = models.resnet18(weights=weights)\n        in_features = model.fc.in_features\n        model.fc = nn.Linear(in_features, num_classes)\n\n    elif "densenet121" in name_clean:\n        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None\n        model = models.densenet121(weights=weights)\n        in_features = model.classifier.in_features\n        model.classifier = nn.Linear(in_features, num_classes)\n\n    elif "efficientnetb0" in name_clean:\n        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None\n        model = models.efficientnet_b0(weights=weights)\n        in_features = model.classifier[1].in_features\n        model.classifier[1] = nn.Linear(in_features, num_classes)\n\n    else:\n        raise ValueError(\n            f"Kiến trúc \'{model_name}\' chưa được hỗ trợ! "\n            f"Các tùy chọn hợp lệ: [\'resnet18\', \'densenet121\', \'efficientnet_b0\']."\n        )\n\n    return model\n\n\ndef freeze_backbone(model: nn.Module) -> None:\n    """Đóng băng toàn bộ trọng số của backbone (chỉ huấn luyện classifier head)."""\n    for param in model.parameters():\n        param.requires_grad = False\n\n    # Mở khóa head phân loại\n    if hasattr(model, "fc"):\n        for param in model.fc.parameters():\n            param.requires_grad = True\n    elif hasattr(model, "classifier"):\n        for param in model.classifier.parameters():\n            param.requires_grad = True\n\n\ndef unfreeze_all(model: nn.Module) -> None:\n    """Mở khóa toàn bộ các tham số trong mạng (fine-tuning toàn bộ mô hình)."""\n    for param in model.parameters():\n        param.requires_grad = True\n')
with open("src/metrics.py", "w", encoding="utf-8") as f: f.write('"""Module metrics.py\n\nTrách nhiệm:\n- Tính toán toàn diện các chỉ số đánh giá cho bài toán phân loại X-quang 4 lớp (phụ trách: TV6).\n- Chỉ số chính: Macro-F1 (tiêu chí chính để chọn mô hình và lưu best checkpoint), Balanced Accuracy.\n- Chỉ số chi tiết: Accuracy, Weighted-F1, Per-class Precision, Per-class Recall, Per-class F1.\n- Xuất ma trận nhầm lẫn (Confusion Matrix) dạng số liệu và biểu đồ trực quan hóa.\n"""\n\nfrom typing import Dict, List, Optional, Union\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n)\n\nDEFAULT_CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]\n\n\ndef compute_metrics(\n    y_true: Union[List[int], np.ndarray],\n    y_pred: Union[List[int], np.ndarray],\n    class_names: Optional[List[str]] = None,\n) -> Dict[str, Union[float, Dict[str, float], List[List[int]]]]:\n    """Tính toán toàn bộ các thước đo hiệu năng phân loại đa lớp.\n\n    Args:\n        y_true: Danh sách hoặc mảng nhãn thực tế (ground truth).\n        y_pred: Danh sách hoặc mảng nhãn mô hình dự đoán.\n        class_names: Danh sách tên các lớp (mặc định 4 lớp).\n\n    Returns:\n        Dict chứa:\n        - \'accuracy\': float\n        - \'balanced_accuracy\': float\n        - \'macro_f1\': float (thước đo chính)\n        - \'weighted_f1\': float\n        - \'macro_precision\': float\n        - \'macro_recall\': float\n        - \'per_class\': Dict tên lớp -> {\'precision\', \'recall\', \'f1\', \'support\'}\n        - \'confusion_matrix\': List[List[int]]\n    """\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n\n    if class_names is None:\n        class_names = DEFAULT_CLASS_NAMES\n    num_classes = len(class_names)\n\n    acc = float(accuracy_score(y_true, y_pred))\n    balanced_acc = float(balanced_accuracy_score(y_true, y_pred))\n    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))\n    weighted_f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))\n    macro_prec = float(precision_score(y_true, y_pred, average="macro", zero_division=0))\n    macro_rec = float(recall_score(y_true, y_pred, average="macro", zero_division=0))\n\n    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))\n\n    # Per-class metrics\n    per_class_prec = precision_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)\n    per_class_rec = recall_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)\n    per_class_f1 = f1_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)\n\n    per_class_dict = {}\n    for idx, name in enumerate(class_names):\n        support_count = int(np.sum(y_true == idx))\n        per_class_dict[name] = {\n            "precision": float(per_class_prec[idx]),\n            "recall": float(per_class_rec[idx]),\n            "f1": float(per_class_f1[idx]),\n            "support": support_count,\n        }\n\n    return {\n        "accuracy": round(acc, 4),\n        "balanced_accuracy": round(balanced_acc, 4),\n        "macro_f1": round(macro_f1, 4),\n        "weighted_f1": round(weighted_f1, 4),\n        "macro_precision": round(macro_prec, 4),\n        "macro_recall": round(macro_rec, 4),\n        "per_class": per_class_dict,\n        "confusion_matrix": cm.tolist(),\n    }\n\n\ndef plot_confusion_matrix(\n    cm: Union[List[List[int]], np.ndarray],\n    class_names: Optional[List[str]] = None,\n    save_path: Optional[str] = None,\n    normalize: bool = False,\n    title: str = "Confusion Matrix",\n) -> plt.Figure:\n    """Vẽ và lưu biểu đồ ma trận nhầm lẫn bằng Matplotlib.\n\n    Args:\n        cm: Ma trận nhầm lẫn dạng 2D array hoặc list.\n        class_names: Danh sách tên lớp hiển thị trên các trục.\n        save_path: Đường dẫn lưu file ảnh (tùy chọn).\n        normalize: True nếu muốn chuẩn hóa tỷ lệ theo hàng (Recall per class).\n        title: Tiêu đề biểu đồ.\n\n    Returns:\n        plt.Figure: Đối tượng Figure của matplotlib.\n    """\n    cm = np.asarray(cm)\n    if class_names is None:\n        class_names = DEFAULT_CLASS_NAMES\n\n    if normalize:\n        row_sums = cm.sum(axis=1)[:, np.newaxis]\n        row_sums[row_sums == 0] = 1\n        cm_display = cm.astype("float") / row_sums\n        fmt = ".2%"\n    else:\n        cm_display = cm\n        fmt = "d"\n\n    fig, ax = plt.subplots(figsize=(7, 6))\n    cax = ax.matshow(cm_display, cmap="Blues")\n    fig.colorbar(cax)\n\n    ax.set_xticks(range(len(class_names)))\n    ax.set_yticks(range(len(class_names)))\n    ax.set_xticklabels(class_names, rotation=30, ha="left", fontsize=9)\n    ax.set_yticklabels(class_names, fontsize=9)\n    ax.set_xlabel("Predicted Label", fontweight="bold", labelpad=10)\n    ax.set_ylabel("True Label", fontweight="bold", labelpad=10)\n    ax.set_title(title, fontweight="bold", pad=15)\n\n    thresh = cm_display.max() / 2.0\n    for i in range(cm.shape[0]):\n        for j in range(cm.shape[1]):\n            val_str = f"{cm_display[i, j]:{fmt}}"\n            color = "white" if cm_display[i, j] > thresh else "black"\n            ax.text(j, i, val_str, ha="center", va="center", color=color, fontsize=9)\n\n    plt.tight_layout()\n    if save_path:\n        plt.savefig(save_path, dpi=150, bbox_inches="tight")\n        print(f"Đã lưu ma trận nhầm lẫn tại: {save_path}")\n\n    return fig\n')
with open("src/train.py", "w", encoding="utf-8") as f: f.write('"""Module train.py\n\nTrách nhiệm:\n- Điều khiển vòng lặp huấn luyện (train loop) và kiểm định (val loop) cho mô hình (phụ trách: TV2).\n- Hỗ trợ huấn luyện độ chính xác hỗn hợp (Mixed Precision) với torch.cuda.amp giúp tối ưu bộ nhớ GPU.\n- Quản lý checkpoint nghiêm ngặt:\n  + Tự động lưu \'last.pt\' sau mỗi epoch để sẵn sàng phục hồi khi kernel bị ngắt phiên (preemption).\n  + Tự động lưu \'best.pt\' khi chỉ số Macro-F1 trên tập Validation đạt kỷ lục mới.\n- Cơ chế Resume Training liền mạch: Nạp model, optimizer, scheduler, scaler và tiếp tục từ epoch tiếp theo.\n- Quy ước đặt tên run chuẩn hóa phục vụ quản lý thực nghiệm.\n"""\n\nfrom datetime import datetime\nimport json\nimport os\nfrom pathlib import Path\nimport time\nfrom typing import Any, Dict, Optional, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\nfrom tqdm.auto import tqdm\n\nfrom src.metrics import compute_metrics\n\n\ndef generate_run_id(model_name: str = "resnet18", seed: int = 42) -> str:\n    """Tạo mã định danh duy nhất cho một lượt huấn luyện (run_id).\n\n    Quy ước đặt tên: <model_name>_seed<seed>_<YYYYMMDD_HHMMSS>\n    Ví dụ: resnet18_seed42_20261002_203000\n    """\n    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")\n    clean_model = model_name.lower().replace("-", "").replace("_", "")\n    return f"{clean_model}_seed{seed}_{timestamp}"\n\n\ndef save_checkpoint(\n    state: Dict[str, Any],\n    filepath: Path,\n) -> None:\n    """Lưu checkpoint PyTorch an toàn vào đĩa."""\n    filepath.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(state, str(filepath))\n\n\ndef load_checkpoint(\n    checkpoint_path: Path,\n    model: nn.Module,\n    optimizer: Optional[torch.optim.Optimizer] = None,\n    scheduler: Optional[Any] = None,\n    scaler: Optional[Any] = None,\n    device: str = "cpu",\n) -> Tuple[int, float]:\n    """Khôi phục trạng thái huấn luyện từ checkpoint .pt để resume phiên làm việc.\n\n    Args:\n        checkpoint_path: Đường dẫn tới file checkpoint (.pt).\n        model: Mô hình PyTorch cần nạp trọng số.\n        optimizer: Bộ tối ưu cần nạp trạng thái (tùy chọn).\n        scheduler: Bộ điều chỉnh learning rate (tùy chọn).\n        scaler: GradScaler của AMP (tùy chọn).\n        device: Thiết bị tính toán (\'cuda\' hoặc \'cpu\').\n\n    Returns:\n        Tuple[int, float]: (start_epoch, best_val_f1)\n        start_epoch: Epoch tiếp theo cần chạy (epoch đã lưu + 1).\n        best_val_f1: Điểm Macro-F1 tốt nhất từng ghi nhận.\n    """\n    if not checkpoint_path.exists():\n        raise FileNotFoundError(f"Không tìm thấy file checkpoint để resume: {checkpoint_path}")\n\n    checkpoint = torch.load(str(checkpoint_path), map_location=device)\n\n    # 1. Nạp trọng số mô hình\n    model.load_state_dict(checkpoint["model_state_dict"])\n\n    # 2. Nạp trạng thái optimizer nếu có\n    if optimizer is not None and "optimizer_state_dict" in checkpoint:\n        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n\n    # 3. Nạp trạng thái scheduler nếu có\n    if scheduler is not None and "scheduler_state_dict" in checkpoint and checkpoint["scheduler_state_dict"] is not None:\n        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])\n\n    # 4. Nạp trạng thái AMP scaler nếu có\n    if scaler is not None and "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:\n        scaler.load_state_dict(checkpoint["scaler_state_dict"])\n\n    saved_epoch = checkpoint.get("epoch", 0)\n    best_val_f1 = checkpoint.get("best_val_f1", checkpoint.get("val_macro_f1", 0.0))\n    start_epoch = saved_epoch + 1\n\n    print(f"ĐÃ KHÔI PHỤC CHECKPOINT: Tiếp tục từ Epoch {start_epoch} (Kỷ lục Macro-F1 trước đó: {best_val_f1:.4f})")\n    return start_epoch, best_val_f1\n\n\ndef train_one_epoch(\n    model: nn.Module,\n    dataloader: DataLoader,\n    criterion: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    scaler: Optional[Any],\n    device: torch.device,\n    use_amp: bool = True,\n) -> Tuple[float, float]:\n    """Thực thi một epoch huấn luyện trên tập Train."""\n    model.train()\n    running_loss = 0.0\n    correct_preds = 0\n    total_samples = 0\n\n    pbar = tqdm(dataloader, desc="Training", leave=False)\n    for images, targets in pbar:\n        images = images.to(device, non_blocking=True)\n        targets = targets.to(device, non_blocking=True)\n\n        optimizer.zero_grad()\n\n        # Mixed Precision context\n        if use_amp and device.type == "cuda" and scaler is not None:\n            with torch.cuda.amp.autocast():\n                outputs = model(images)\n                loss = criterion(outputs, targets)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n        else:\n            outputs = model(images)\n            loss = criterion(outputs, targets)\n            loss.backward()\n            optimizer.step()\n\n        batch_size = images.size(0)\n        running_loss += loss.item() * batch_size\n        _, preds = torch.max(outputs, 1)\n        correct_preds += torch.sum(preds == targets.data).item()\n        total_samples += batch_size\n\n        pbar.set_postfix({"loss": f"{loss.item():.4f}"})\n\n    epoch_loss = running_loss / total_samples\n    epoch_acc = correct_preds / total_samples\n    return epoch_loss, epoch_acc\n\n\ndef validate(\n    model: nn.Module,\n    dataloader: DataLoader,\n    criterion: nn.Module,\n    device: torch.device,\n    class_names: Optional[list] = None,\n) -> Tuple[float, Dict[str, Any]]:\n    """Thực thi đánh giá trên tập Validation."""\n    model.eval()\n    running_loss = 0.0\n    total_samples = 0\n    all_targets = []\n    all_preds = []\n\n    with torch.no_grad():\n        for images, targets in tqdm(dataloader, desc="Validating", leave=False):\n            images = images.to(device, non_blocking=True)\n            targets = targets.to(device, non_blocking=True)\n\n            outputs = model(images)\n            loss = criterion(outputs, targets)\n\n            batch_size = images.size(0)\n            running_loss += loss.item() * batch_size\n            total_samples += batch_size\n\n            _, preds = torch.max(outputs, 1)\n            all_targets.extend(targets.cpu().numpy())\n            all_preds.extend(preds.cpu().numpy())\n\n    val_loss = running_loss / total_samples\n    val_metrics = compute_metrics(all_targets, all_preds, class_names=class_names)\n    return val_loss, val_metrics\n\n\ndef train_pipeline(\n    model: nn.Module,\n    train_loader: DataLoader,\n    val_loader: DataLoader,\n    criterion: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    scheduler: Optional[Any] = None,\n    num_epochs: int = 5,\n    device_name: str = "cuda" if torch.cuda.is_available() else "cpu",\n    output_dir: Path = Path("outputs/checkpoints"),\n    run_id: Optional[str] = None,\n    resume_from: Optional[Path] = None,\n    use_amp: bool = True,\n    class_names: Optional[list] = None,\n) -> Dict[str, Any]:\n    """Pipeline điều phối toàn bộ quá trình huấn luyện và lưu checkpoint."""\n    device = torch.device(device_name)\n    model.to(device)\n\n    if run_id is None:\n        run_id = generate_run_id()\n\n    checkpoint_dir = output_dir / run_id\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n\n    last_pt_path = checkpoint_dir / "last.pt"\n    best_pt_path = checkpoint_dir / "best.pt"\n    history_path = checkpoint_dir / "train_history.json"\n\n    # Cấu hình AMP GradScaler\n    scaler = None\n    if use_amp and device.type == "cuda":\n        scaler = torch.cuda.amp.GradScaler()\n\n    start_epoch = 1\n    best_val_f1 = 0.0\n    history = []\n\n    # Xử lý Resume\n    if resume_from is not None and Path(resume_from).exists():\n        start_epoch, best_val_f1 = load_checkpoint(\n            checkpoint_path=Path(resume_from),\n            model=model,\n            optimizer=optimizer,\n            scheduler=scheduler,\n            scaler=scaler,\n            device=device_name,\n        )\n        if history_path.exists():\n            with open(history_path, "r", encoding="utf-8") as f:\n                history = json.load(f)\n\n    print(f"=== BẮT ĐẦU HUẤN LUYỆN: Run ID = {run_id} | Thiết bị = {device} | Epochs = {start_epoch} -> {num_epochs} ===")\n\n    for epoch in range(start_epoch, num_epochs + 1):\n        t0 = time.time()\n        print(f"\\n--- Epoch {epoch}/{num_epochs} ---")\n\n        train_loss, train_acc = train_one_epoch(\n            model=model,\n            dataloader=train_loader,\n            criterion=criterion,\n            optimizer=optimizer,\n            scaler=scaler,\n            device=device,\n            use_amp=use_amp,\n        )\n\n        val_loss, val_metrics = validate(\n            model=model,\n            dataloader=val_loader,\n            criterion=criterion,\n            device=device,\n            class_names=class_names,\n        )\n\n        if scheduler is not None:\n            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):\n                scheduler.step(val_loss)\n            else:\n                scheduler.step()\n\n        elapsed = time.time() - t0\n        val_f1 = val_metrics["macro_f1"]\n        val_acc = val_metrics["accuracy"]\n\n        print(\n            f"Epoch {epoch:02d} [{elapsed:.1f}s]: "\n            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "\n            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}"\n        )\n\n        # Trạng thái checkpoint\n        ckpt_state = {\n            "epoch": epoch,\n            "run_id": run_id,\n            "model_state_dict": model.state_dict(),\n            "optimizer_state_dict": optimizer.state_dict(),\n            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,\n            "scaler_state_dict": scaler.state_dict() if scaler else None,\n            "train_loss": train_loss,\n            "val_loss": val_loss,\n            "val_macro_f1": val_f1,\n            "best_val_f1": max(best_val_f1, val_f1),\n        }\n\n        # 1. Luôn lưu last.pt phục vụ resume\n        save_checkpoint(ckpt_state, last_pt_path)\n\n        # 2. Lưu best.pt khi val_macro_f1 cải thiện\n        is_best = val_f1 > best_val_f1\n        if is_best:\n            best_val_f1 = val_f1\n            ckpt_state["best_val_f1"] = best_val_f1\n            save_checkpoint(ckpt_state, best_pt_path)\n            print(f"⭐ KỶ LỤC MỚI: Đã cập nhật best.pt (Macro-F1 = {best_val_f1:.4f})")\n\n        # Lưu lịch sử log\n        epoch_record = {\n            "epoch": epoch,\n            "train_loss": round(train_loss, 4),\n            "train_acc": round(train_acc, 4),\n            "val_loss": round(val_loss, 4),\n            "val_acc": round(val_acc, 4),\n            "val_macro_f1": round(val_f1, 4),\n            "is_best": is_best,\n            "elapsed_seconds": round(elapsed, 2),\n        }\n        history.append(epoch_record)\n        with open(history_path, "w", encoding="utf-8") as f:\n            json.dump(history, f, indent=2, ensure_ascii=False)\n\n    print(f"\\nHUẤN LUYỆN HOÀN TẤT. Kỷ lục Macro-F1 đạt được: {best_val_f1:.4f}")\n    return {\n        "run_id": run_id,\n        "best_val_f1": best_val_f1,\n        "last_checkpoint": str(last_pt_path),\n        "best_checkpoint": str(best_pt_path),\n        "history": history,\n    }\n')

with open("tests/test_split.py", "w", encoding="utf-8") as f: f.write('"""Bộ kiểm thử tính toàn vẹn và chống rò rỉ của tập dữ liệu phân chia (splits/split_v1.csv).\n\nTuần 3 (Sprint 2) - Track A.\nPhụ trách: TV1 (kèm cặp: TV2).\n\nKiểm tra 4 điều kiện cốt lõi:\n1. Không rò rỉ nhóm (Leakage Prevention): Tập group_id giữa train, val, test rời nhau hoàn toàn.\n2. Tỷ lệ phân chia xấp xỉ 70/15/15: Tổng số mẫu đúng 21.165, tỷ lệ train/val/test nằm trong dung sai cho phép.\n3. Phân bố lớp đồng đều (Stratification): Tỷ lệ từng lớp trong train, val, test khớp với tỷ lệ tổng thể.\n4. Toàn vẹn và mã băm SHA256 (Integrity): Mã băm của split_v1.csv khớp chính xác với file split_v1.sha256.\n"""\n\nimport csv\nimport hashlib\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nREPO_ROOT = Path(__file__).resolve().parent.parent\nSPLIT_CSV_PATH = REPO_ROOT / "splits" / "split_v1.csv"\nSPLIT_SHA_PATH = REPO_ROOT / "splits" / "split_v1.sha256"\n\nEXPECTED_TOTAL_ROWS = 21165\nEXPECTED_CLASSES = {"COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"}\nREQUIRED_COLUMNS = {"image_id", "label", "rel_image_path", "group_id", "split"}\n\n\ndef load_split_records():\n    assert SPLIT_CSV_PATH.exists(), f"Không tìm thấy file: {SPLIT_CSV_PATH}"\n    records = []\n    with open(SPLIT_CSV_PATH, "r", encoding="utf-8") as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n            records.append(row)\n    return records\n\n\ndef test_file_existence_and_columns():\n    """Kiểm tra sự tồn tại của file và các cột bắt buộc."""\n    assert SPLIT_CSV_PATH.exists(), "splits/split_v1.csv chưa tồn tại!"\n    assert SPLIT_SHA_PATH.exists(), "splits/split_v1.sha256 chưa tồn tại!"\n\n    records = load_split_records()\n    assert len(records) == EXPECTED_TOTAL_ROWS, (\n        f"Tổng số ảnh ({len(records)}) không khớp với kỳ vọng ({EXPECTED_TOTAL_ROWS})"\n    )\n\n    first_row = records[0]\n    for col in REQUIRED_COLUMNS:\n        assert col in first_row, f"Thiếu cột bắt buộc: {col}"\n\n\ndef test_no_group_leakage():\n    """Điều kiện 1: Không rò rỉ nhóm trùng lặp giữa các tập train, val, test."""\n    records = load_split_records()\n    split_groups = defaultdict(set)\n    for r in records:\n        split_groups[r["split"]].add(r["group_id"])\n\n    train_groups = split_groups["train"]\n    val_groups = split_groups["val"]\n    test_groups = split_groups["test"]\n\n    train_val_overlap = train_groups & val_groups\n    train_test_overlap = train_groups & test_groups\n    val_test_overlap = val_groups & test_groups\n\n    assert len(train_val_overlap) == 0, (\n        f"Rò rỉ nhóm giữa train và val: {len(train_val_overlap)} nhóm trùng!"\n    )\n    assert len(train_test_overlap) == 0, (\n        f"Rò rỉ nhóm giữa train và test: {len(train_test_overlap)} nhóm trùng!"\n    )\n    assert len(val_test_overlap) == 0, (\n        f"Rò rỉ nhóm giữa val và test: {len(val_test_overlap)} nhóm trùng!"\n    )\n\n\ndef test_split_ratios():\n    """Điều kiện 2: Tỷ lệ xấp xỉ 70% train, 15% val, 15% test (dung sai +/- 3%)."""\n    records = load_split_records()\n    total = len(records)\n    split_counts = Counter(r["split"] for r in records)\n\n    train_count = split_counts["train"]\n    val_count = split_counts["val"]\n    test_count = split_counts["test"]\n\n    assert train_count + val_count + test_count == total\n\n    train_ratio = train_count / total\n    val_ratio = val_count / total\n    test_ratio = test_count / total\n\n    # Cho phép dung sai 3% do ràng buộc gom nhóm group_id\n    assert 0.67 <= train_ratio <= 0.73, f"Tỷ lệ train ngoài khoảng [67%, 73%]: {train_ratio:.3f}"\n    assert 0.12 <= val_ratio <= 0.18, f"Tỷ lệ val ngoài khoảng [12%, 18%]: {val_ratio:.3f}"\n    assert 0.12 <= test_ratio <= 0.18, f"Tỷ lệ test ngoài khoảng [12%, 18%]: {test_ratio:.3f}"\n\n\ndef test_class_distribution_balance():\n    """Điều kiện 3: Phân bố các lớp đồng đều và phân tầng trên từng split."""\n    records = load_split_records()\n    total = len(records)\n\n    global_class_counts = Counter(r["label"] for r in records)\n    assert set(global_class_counts.keys()) == EXPECTED_CLASSES, (\n        f"Các lớp không khớp: {set(global_class_counts.keys())}"\n    )\n\n    split_class_counts = defaultdict(Counter)\n    for r in records:\n        split_class_counts[r["split"]][r["label"]] += 1\n\n    # Kiểm tra tỷ lệ từng lớp trong từng split so với tỷ lệ toàn cục (độ lệch tối đa < 3%)\n    for split_name in ["train", "val", "test"]:\n        split_total = sum(split_class_counts[split_name].values())\n        for cls, count in global_class_counts.items():\n            global_prop = count / total\n            split_prop = split_class_counts[split_name][cls] / split_total\n            diff = abs(split_prop - global_prop)\n            assert diff < 0.035, (\n                f"Lệch phân bố lớp {cls} ở tập {split_name}: {split_prop:.3f} vs toàn cục {global_prop:.3f} (diff={diff:.3f})"\n            )\n\n\ndef test_sha256_integrity():\n    """Điều kiện 4: Mã băm SHA256 của split_v1.csv phải khớp tuyệt đối với file split_v1.sha256."""\n    assert SPLIT_SHA_PATH.exists(), f"Không tìm thấy file mã băm: {SPLIT_SHA_PATH}"\n\n    sha_content = SPLIT_SHA_PATH.read_text(encoding="utf-8").strip()\n    expected_hash = sha_content.split()[0].lower()\n\n    hasher = hashlib.sha256()\n    with open(SPLIT_CSV_PATH, "rb") as f:\n        for chunk in iter(lambda: f.read(65536), b""):\n            hasher.update(chunk)\n    actual_hash = hasher.hexdigest().lower()\n\n    assert actual_hash == expected_hash, (\n        f"Mã SHA256 không khớp!\\nKỳ vọng: {expected_hash}\\nThực tế: {actual_hash}"\n    )\n\n\nif __name__ == "__main__":\n    print("Đang kiểm tra tính toàn vẹn của split...")\n    test_file_existence_and_columns()\n    test_no_group_leakage()\n    test_split_ratios()\n    test_class_distribution_balance()\n    test_sha256_integrity()\n    print("Tất cả các bài kiểm tra đều ĐẠT!")\n')
with open("tests/test_metrics.py", "w", encoding="utf-8") as f: f.write('"""Kiểm thử đơn vị cho module src/metrics.py.\n\nTuần 3 (Sprint 2) - Track B.\nPhụ trách: TV6.\n"""\n\nimport numpy as np\nimport pytest\nfrom src.metrics import compute_metrics, plot_confusion_matrix, DEFAULT_CLASS_NAMES\n\n\ndef test_compute_metrics_perfect_prediction():\n    """Kiểm tra trường hợp dự đoán hoàn hảo 100%."""\n    y_true = [0, 1, 2, 3, 0, 1, 2, 3]\n    y_pred = [0, 1, 2, 3, 0, 1, 2, 3]\n\n    res = compute_metrics(y_true, y_pred)\n    assert res["accuracy"] == 1.0\n    assert res["balanced_accuracy"] == 1.0\n    assert res["macro_f1"] == 1.0\n    assert res["weighted_f1"] == 1.0\n    assert res["macro_precision"] == 1.0\n    assert res["macro_recall"] == 1.0\n\n    # Kiểm tra per_class\n    for cls in DEFAULT_CLASS_NAMES:\n        assert res["per_class"][cls]["f1"] == 1.0\n        assert res["per_class"][cls]["support"] == 2\n\n\ndef test_compute_metrics_all_wrong():\n    """Kiểm tra trường hợp dự đoán sai hoàn toàn."""\n    y_true = [0, 0, 0, 0]\n    y_pred = [1, 1, 1, 1]\n\n    res = compute_metrics(y_true, y_pred)\n    assert res["accuracy"] == 0.0\n    assert res["balanced_accuracy"] == 0.0\n\n\ndef test_compute_metrics_partial_accuracy():\n    """Kiểm tra tính toán với ma trận nhầm lẫn đã biết."""\n    # 2 mẫu lớp 0 (1 đúng, 1 sai thành 1)\n    # 2 mẫu lớp 1 (cả 2 đúng)\n    y_true = [0, 0, 1, 1]\n    y_pred = [0, 1, 1, 1]\n\n    res = compute_metrics(y_true, y_pred, class_names=["COVID", "Normal"])\n    # Accuracy = 3/4 = 0.75\n    assert res["accuracy"] == 0.75\n\n    # Recall lớp 0 = 1/2 = 0.5; Recall lớp 1 = 2/2 = 1.0 -> Balanced Acc = (0.5 + 1.0)/2 = 0.75\n    assert res["balanced_accuracy"] == 0.75\n\n    # Precision lớp 0 = 1/1 = 1.0; Precision lớp 1 = 2/3 = 0.6667\n    assert res["per_class"]["COVID"]["precision"] == 1.0\n    assert round(res["per_class"]["Normal"]["precision"], 4) == 0.6667\n\n\ndef test_plot_confusion_matrix():\n    """Kiểm tra hàm vẽ ma trận nhầm lẫn chạy không sinh lỗi."""\n    cm = [[10, 2], [1, 15]]\n    fig = plot_confusion_matrix(cm, class_names=["COVID", "Normal"], normalize=True)\n    assert fig is not None\n\n\nif __name__ == "__main__":\n    test_compute_metrics_perfect_prediction()\n    test_compute_metrics_all_wrong()\n    test_compute_metrics_partial_accuracy()\n    test_plot_confusion_matrix()\n    print("Tất cả bài kiểm tra metrics đều ĐẠT!")\n')
with open("tests/test_resume.py", "w", encoding="utf-8") as f: f.write('"""Kiểm thử đơn vị cho cơ chế checkpoint / resume và cổng khóa tập Test.\n\nTuần 3 (Sprint 2) - Track B.\nPhụ trách: TV2 (kèm cặp: TV1).\n"""\n\nfrom pathlib import Path\nimport tempfile\nimport torch\nimport torch.nn as nn\nimport pytest\n\nfrom src.train import save_checkpoint, load_checkpoint\nfrom src.dataset import CXRDataset\n\n\nclass DummyModel(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.fc = nn.Linear(8, 4)\n\n    def forward(self, x):\n        return self.fc(x)\n\n\ndef test_checkpoint_save_and_resume():\n    """Kiểm tra lưu checkpoint và resume chính xác epoch tiếp theo."""\n    with tempfile.TemporaryDirectory() as tmpdir:\n        ckpt_path = Path(tmpdir) / "last.pt"\n\n        # 1. Khởi tạo mô hình và optimizer ban đầu\n        model_orig = DummyModel()\n        optimizer_orig = torch.optim.Adam(model_orig.parameters(), lr=0.01)\n\n        # Chạy 1 bước giả lập để optimizer có state\n        dummy_x = torch.randn(2, 8)\n        dummy_y = torch.tensor([0, 1])\n        loss = nn.CrossEntropyLoss()(model_orig(dummy_x), dummy_y)\n        loss.backward()\n        optimizer_orig.step()\n\n        # Lưu checkpoint tại epoch 2\n        state = {\n            "epoch": 2,\n            "model_state_dict": model_orig.state_dict(),\n            "optimizer_state_dict": optimizer_orig.state_dict(),\n            "best_val_f1": 0.8542,\n        }\n        save_checkpoint(state, ckpt_path)\n        assert ckpt_path.exists()\n\n        # 2. Khởi tạo mô hình mới và resume\n        model_resumed = DummyModel()\n        optimizer_resumed = torch.optim.Adam(model_resumed.parameters(), lr=0.01)\n\n        start_epoch, best_val_f1 = load_checkpoint(\n            checkpoint_path=ckpt_path,\n            model=model_resumed,\n            optimizer=optimizer_resumed,\n        )\n\n        # Kiểm tra start_epoch phải là epoch tiếp theo (2 + 1 = 3)\n        assert start_epoch == 3\n        assert best_val_f1 == 0.8542\n\n        # Kiểm tra trọng số mô hình khớp chính xác 100%\n        for k in model_orig.state_dict():\n            assert torch.equal(model_orig.state_dict()[k], model_resumed.state_dict()[k])\n\n\ndef test_resume_non_existent_file():\n    """Kiểm tra ném FileNotFoundError khi file checkpoint không tồn tại."""\n    model = DummyModel()\n    with pytest.raises(FileNotFoundError):\n        load_checkpoint(Path("non_existent_file.pt"), model)\n\n\ndef test_dataset_allow_test_lock():\n    """Kiểm tra cơ chế khóa tập test: ném PermissionError khi allow_test=False."""\n    import pandas as pd\n\n    # Tạo dataframe giả lập\n    dummy_df = pd.DataFrame({\n        "image_id": ["dummy.png"],\n        "label": ["COVID"],\n        "rel_image_path": ["dummy.png"],\n        "split": ["test"]\n    })\n\n    # Cố tình truy cập split=\'test\' khi allow_test=False -> BẮT BUỘC ném lỗi PermissionError\n    with pytest.raises(PermissionError):\n        CXRDataset(split_df_or_path=dummy_df, split="test", allow_test=False)\n\n\nif __name__ == "__main__":\n    test_checkpoint_save_and_resume()\n    test_resume_non_existent_file()\n    test_dataset_allow_test_lock()\n    print("Tất cả bài kiểm tra resume và test locking đều ĐẠT!")\n')

if "." not in sys.path:
    sys.path.insert(0, ".")

print("Đã thiết lập cấu trúc mã nguồn src/ và tests/ thành công!")


In [ ]:
# 2. Kiểm tra thông số hệ thống và GPU
import torch
import torchvision
import numpy as np
import pandas as pd

print("=== THÔNG TIN HỆ THỐNG ===")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Count:", torch.cuda.device_count())
    print("GPU 0 Name:", torch.cuda.get_device_name(0))
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
    print("CẢNH BÁO: Đang chạy trên CPU!")


In [ ]:
# 3. Định vị split_v1.csv và thư mục ảnh
import shutil
input_root = Path("/kaggle/input")
if not input_root.exists():
    input_root = Path(".")

# Tìm split_v1.csv
split_csv_path = None
for cand in [Path("splits/split_v1.csv"), Path("/kaggle/input/cxr-duplicates-and-split/split_v1.csv")]:
    if cand.exists():
        split_csv_path = cand
        break

if split_csv_path is None:
    for p in input_root.rglob("split_v1.csv"):
        split_csv_path = p
        break

os.makedirs("splits", exist_ok=True)
if split_csv_path and not Path("splits/split_v1.csv").exists():
    shutil.copy(split_csv_path, "splits/split_v1.csv")
    
sha_cand = split_csv_path.parent / "split_v1.sha256" if split_csv_path else None
if sha_cand and sha_cand.exists() and not Path("splits/split_v1.sha256").exists():
    shutil.copy(sha_cand, "splits/split_v1.sha256")

print(f"File split_v1.csv: {Path('splits/split_v1.csv').resolve()}")

# Tìm data_root chứa các thư mục ảnh (COVID, Normal,...)
data_root = None
for d in [input_root] + [p for p in input_root.rglob("*") if p.is_dir()]:
    if (d / "COVID").is_dir() and ((d / "COVID" / "images").is_dir() or (d / "Normal").is_dir()):
        data_root = d
        break
print(f"Thư mục data_root xác định được: {data_root}")


In [ ]:
# 4. Chạy bộ kiểm thử tự động (pytest)
print("=== CHẠY KIỂM THỬ ĐƠN VỊ TỰ ĐỘNG ===")
!pytest tests/ -v


In [ ]:
# 5. Kiểm tra cơ chế khóa tập Test (allow_test=False)
from src.dataset import CXRDataset, compute_class_weights, build_dataloaders

print("Kiểm tra bảo vệ tập Test...")
try:
    test_ds = CXRDataset(split_df_or_path="splits/split_v1.csv", split="test", allow_test=False)
    raise AssertionError("LỖI: Truy cập tập test trái phép mà không bị chặn!")
except PermissionError as e:
    print(" XÁC NHẬN BẢO VỆ: Đã kích hoạt PermissionError thành công khi cố truy cập tập test!")
    print("Thông điệp cảnh báo:", str(e).split('\n')[0])

# Tính class weights
class_weights = compute_class_weights("splits/split_v1.csv", split="train")
print(f"\nTrọng số lớp đã tính (Inverse Frequency): {class_weights.tolist()}")


In [ ]:
# 6. Chuẩn bị DataLoader cho Smoke Test (Subset nhỏ để kiểm tra tính năng nhanh)
from torch.utils.data import Subset, DataLoader
from src.models import get_model
from src.train import train_pipeline, load_checkpoint, generate_run_id

full_loaders = build_dataloaders(
    split_csv_path="splits/split_v1.csv",
    data_root=data_root,
    batch_size=16,
    num_workers=2,
    img_size=224,
    allow_test=False
)

# Lấy subset 64 mẫu train và 32 mẫu val để thực thi smoke test nhanh (<1 phút)
train_subset = Subset(full_loaders["train"].dataset, list(range(64)))
val_subset = Subset(full_loaders["val"].dataset, list(range(32)))

smoke_train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=2)
smoke_val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=2)

print(f"Smoke loaders đã sẵn sàng: Train={len(train_subset)} mẫu | Val={len(val_subset)} mẫu")


In [ ]:
# 7. Chạy Epoch 1 (Giai đoạn 1: Tạo checkpoint last.pt ban đầu)
run_id = generate_run_id("resnet18_smoke", seed=42)
print(f"Khởi tạo Smoke Run ID: {run_id}")

# Khởi tạo ResNet-18 (pretrained=False, không cần internet)
model = get_model("resnet18", num_classes=4, pretrained=False)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

ckpt_dir = Path("/kaggle/working/outputs/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

print("=== BẮT ĐẦU CHẠY EPOCH 1 ===")
res1 = train_pipeline(
    model=model,
    train_loader=smoke_train_loader,
    val_loader=smoke_val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=1,
    device_name="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=ckpt_dir,
    run_id=run_id,
    use_amp=True
)

last_pt = Path(res1["last_checkpoint"])
assert last_pt.exists(), f"LỖI: Chưa tạo được last.pt tại {last_pt}"
print(f" Epoch 1 hoàn tất thành công. File last.pt đã được tạo: {last_pt} ({last_pt.stat().st_size / 1024 / 1024:.2f} MB)")


In [ ]:
# 8. Giả lập Ngắt Phiên & Kiểm chứng Cơ Chế Resume tại Epoch 2
print("=== GIẢ LẬP SỰ CỐ NGẮT PHIÊN (PREEMPTION SIMULATION) ===")
# Hủy bỏ các đối tượng mô hình trong bộ nhớ để giả lập phiên làm việc mới
del model
del optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Đã giải phóng bộ nhớ mô hình cũ.")

print("\n=== KHỞI TẠO LẠI VÀ THỰC HIỆN RESUME TỪ last.pt ===")
new_model = get_model("resnet18", num_classes=4, pretrained=False)
new_optimizer = torch.optim.AdamW(new_model.parameters(), lr=1e-3, weight_decay=1e-4)

res2 = train_pipeline(
    model=new_model,
    train_loader=smoke_train_loader,
    val_loader=smoke_val_loader,
    criterion=criterion,
    optimizer=new_optimizer,
    num_epochs=2,  # Đích đến là epoch 2
    device_name="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=ckpt_dir,
    run_id=run_id,
    resume_from=last_pt,
    use_amp=True
)

history = res2["history"]
print(f"\nLịch sử huấn luyện sau khi Resume ({len(history)} epochs):")
for h in history:
    print(f" - Epoch {h['epoch']}: Train Loss = {h['train_loss']}, Val Macro-F1 = {h['val_macro_f1']}")

assert len(history) == 2, f"LỖI: Lịch sử phải đủ 2 epochs, hiện có {len(history)}"
assert history[1]["epoch"] == 2, "LỖI: Epoch tiếp theo phải là Epoch 2"
print(" XÁC MINH THÀNH CÔNG: Cơ chế Resume hoạt động chính xác từ epoch 1 -> 2!")


In [ ]:
# 9. Xuất báo cáo nghiệm thu Smoke Test JSON ra /kaggle/working
import json

report = {
    "status": "SUCCESS",
    "device": str(device),
    "cuda_available": torch.cuda.is_available(),
    "model_tested": "resnet18 (pretrained=False)",
    "mixed_precision_amp": True,
    "resume_verified": True,
    "test_locked_verified": True,
    "last_checkpoint_size_mb": round(last_pt.stat().st_size / 1024 / 1024, 2),
    "epochs_completed": 2,
    "final_val_macro_f1": history[-1]["val_macro_f1"],
}

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
out_report = working_dir / "smoke_test_report.json"
with open(out_report, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Đã xuất smoke_test_report.json tại: {out_report.resolve()}")
print("=== TOÀN BỘ TIÊU CHÍ TRACK B ĐÃ HOÀN TẤT & ĐẠT CHUẨN ===")
